#  First Machine Learning Classifier

## 1. The prediction
**Predict whether a Titanic passenger survived, using their class, sex, age and fare.**
- **Target (y):** `survived` (0 = died, 1 = survived)
- **Features (X):** `pclass`, `sex`, `age`, `fare`



In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             mean_squared_error)

sns.set_theme(style="whitegrid", palette="colorblind")

## 2. Prepare the data


In [ ]:
raw = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age", "fare"]]
print("Rows before cleaning:", len(raw))
print("Missing values per column:")
print(raw.isna().sum().to_string())

df = raw.dropna().copy()
df["sex"] = (df["sex"] == "female").astype(int)   # female = 1, male = 0
print("\nRows after dropping missing values:", len(df))
df.head()

In [ ]:
X = df.drop(columns="survived")
y = df["survived"]
print("Overall survival rate:", round(y.mean(), 3))

## 3. Split into training and test sets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Survival rate - train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

## 4. Baseline


In [ ]:
baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Baseline accuracy (always guess the most common outcome): {baseline:.3f}")

## 5. Train the decision tree


In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

## 6. Evaluate on the test set

In [ ]:
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
cm = confusion_matrix(y_test, pred)

print(f"Model accuracy on the test set: {acc:.3f}")
print(f"Baseline accuracy:              {baseline:.3f}")
print(f"Improvement over baseline:      {acc - baseline:+.3f}")
print()
print(classification_report(y_test, pred, target_names=["Died", "Survived"]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Predicted died", "Predicted survived"],
            yticklabels=["Actually died", "Actually survived"])
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives  (died, predicted died):         {tn}")
print(f"False positives (died, predicted survived):     {fp}")
print(f"False negatives (survived, predicted died):     {fn}")
print(f"True positives  (survived, predicted survived): {tp}")

**How to read the four cells**
- **True negative:** the passenger died and the model said died (correct).
- **False positive:** the passenger died but the model said survived (wrong).
- **False negative:** the passenger survived but the model said died (wrong).
- **True positive:** the passenger survived and the model said survived (correct).



## 7. Depth, overfitting and the test set


In [ ]:
depths = range(1, 13)
train_acc, test_acc = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc.append(m.score(X_train, y_train))
    test_acc.append(m.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths, train_acc, marker="o", label="Training accuracy")
ax.plot(depths, test_acc, marker="o", label="Test accuracy")
ax.axhline(baseline, linestyle="--", color="grey", label="Baseline")
ax.set(title="Deeper trees fit the training data better but not new data",
       xlabel="Tree depth (max_depth)", ylabel="Accuracy")
ax.legend()
plt.tight_layout()
plt.show()

best_d = list(depths)[int(np.argmax(test_acc))]
print(f"Depth with the highest test accuracy: {best_d} ({max(test_acc):.3f})")
print(f"At depth 12: training accuracy {train_acc[-1]:.3f} vs test accuracy {test_acc[-1]:.3f}")

**Why very deep trees do worse on new data (overfitting):** a very deep tree keeps splitting until it memorises the training passengers, including their noise and odd cases. Its training accuracy approaches 100%, but those memorised patterns do not hold for new passengers, so test accuracy stops improving or drops. A shallower tree learns only the broad patterns, which generalise better.

**Caution:** choosing the depth by looking at test accuracy uses the test set to make a decision, so the test score becomes slightly optimistic. With more time I would use cross-validation instead.

## 8. Which features mattered?

In [ ]:
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=imp.values, y=imp.index, ax=ax)
ax.set(title=f"{imp.index[0]} is the most important feature", xlabel="Importance", ylabel="Feature")
plt.tight_layout()
plt.show()

print(imp.round(2).to_string())

## 9. Bonus: MSE with a Linear Regression model


In [ ]:
reg = df[["pclass", "sex", "age", "fare"]]
Xr, yr = reg.drop(columns="fare"), reg["fare"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)

lin = LinearRegression().fit(Xr_train, yr_train)
mse = mean_squared_error(yr_test, lin.predict(Xr_test))
mse_base = mean_squared_error(yr_test, np.full(len(yr_test), yr_train.mean()))

print(f"Linear Regression MSE: {mse:,.1f} (RMSE ${np.sqrt(mse):,.1f})")
print(f"Baseline MSE (always predict the mean fare): {mse_base:,.1f} (RMSE ${np.sqrt(mse_base):,.1f})")
print(f"R-squared on the test set: {lin.score(Xr_test, yr_test):.2f}")

Lower MSE is better. Fares are skewed (a few very expensive tickets), so squared errors are dominated by those large fares, and this simple linear model leaves a lot unexplained.

## 10. What I learned and limitations
- The decision tree beats the baseline on unseen data, so it learned real patterns (see the numbers above).
- The feature importance chart shows which columns the tree relied on most; sex is expected to lead, in line with the survival gap found in Project 1 (check the chart for the actual ranking).
- Training accuracy keeps rising with depth while test accuracy does not, which shows overfitting.

**Limitations**
- **Small data:** after dropping missing ages, only a few hundred passengers remain and the test set is about 140 people, so the accuracy is noisy and could change with a different split.
- **Dropped rows:** removing passengers with missing age may bias the data, as the missing ages are not random.
- **Few features:** I used only four columns; family size, title or port might help.
- **One split:** a single train/test split gives one estimate; cross-validation would be more reliable.
- **Association, not cause:** the model finds patterns in who survived, not why.

In [ ]:
top2 = imp.index[:2].tolist()
note = (
    f"I trained a decision tree (max_depth=4) on the Titanic dataset to predict survival from pclass, sex, age and fare, "
    f"using {len(df)} passengers after dropping rows with missing values and an 80/20 stratified split "
    f"(random_state=42). On the unseen test set it reached {acc:.1%} accuracy against a baseline of {baseline:.1%} "
    f"(always guessing the most common outcome). The confusion matrix shows {tp} true positives, {tn} true negatives, "
    f"{fp} false positives and {fn} false negatives. Testing depths 1 to 12 showed overfitting: at depth 12 training "
    f"accuracy was {train_acc[-1]:.1%} but test accuracy only {test_acc[-1]:.1%}. The most important features were "
    f"{top2[0]} and {top2[1]}. As a bonus, a linear regression predicting fare had an MSE of {mse:,.0f} versus "
    f"{mse_base:,.0f} for the mean baseline. Limitations: a small dataset with a noisy test set, dropped rows with "
    f"missing age, few features and a single train/test split."
)
print(note)